In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Data and Preprocessing

In [ ]:
# ================= CONFIG =================
# ================= CONFIG =================
SEED = 42
WINDOW = 24
K_NEIGHBORS = 4

# Fault belief threshold (control-level, not detection)
p_fault_threshold = 0.6


import numpy as np
import pandas as pd
import torch
import random

np.random.seed(SEED)
torch.manual_seed(SEED)
random.seed(SEED)


In [ ]:
# ================= DATA LOAD =================
df_wide = pd.read_parquet(
    "/kaggle/input/baseline-artifacts/df_wide.parquet"
)

df_wide = df_wide.sort_index()
sensor_ids = df_wide.columns.tolist()

T, N = df_wide.shape
print("Time steps:", T)
print("Sensors:", N)


In [ ]:
# ================= MISSING DATA =================
df_filled = df_wide.interpolate(limit_direction="both")
df_filled = df_filled.fillna(df_filled.mean())

assert not df_filled.isna().any().any(), "❌ NaNs still present"


In [ ]:
# ================= NORMALIZATION =================
X_np = df_filled.values.astype(np.float32)

mean = X_np.mean()
std  = X_np.std() + 1e-6

X_np = (X_np - mean) / std
X = torch.tensor(X_np, dtype=torch.float32)

assert torch.isfinite(X).all(), "❌ Non-finite values"


In [ ]:
# ================= SENSOR METADATA =================
import geopandas as gpd

DBF_PATH = "/kaggle/input/caf-dataset/CAF_sensors.dbf"
gdf = gpd.read_file(DBF_PATH)

sensor_meta = gdf[["Location", "Easting", "Northing"]].copy()

coords = (
    sensor_meta
    .set_index("Location")
    .loc[sensor_ids][["Easting", "Northing"]]
    .values
)

assert coords.shape == (N, 2)


In [ ]:
# ================= SPATIAL GRAPH =================
from sklearn.neighbors import NearestNeighbors

nbrs = NearestNeighbors(n_neighbors=K_NEIGHBORS + 1).fit(coords)
distances, indices = nbrs.kneighbors(coords)

# Remove self-loops (first neighbor)
neighbors = indices[:, 1:]
neighbor_dist = distances[:, 1:]


In [ ]:
# ================= NEIGHBOR HEALTH TRUST =================
# health_j = how often sensor j agrees with its neighbors (long-term)

X_np = X.cpu().numpy()   # CLEAN, normalized data

health = np.zeros(N)

for j in range(N):
    nbrs = neighbors[j]
    nbr_median = np.median(X_np[:, nbrs], axis=1)
    health[j] = np.exp(
        -np.median(np.abs(X_np[:, j] - nbr_median))
    )

# normalize to [0,1]
health = health / (health.max() + 1e-6)
health = torch.tensor(health, dtype=torch.float32)


In [ ]:
# ================= DISTANCE-WEIGHTED ADJ =================
A = np.zeros((N, N), dtype=np.float32)

for i in range(N):
    for j, d in zip(neighbors[i], neighbor_dist[i]):
        A[i, j] = 1.0 / (d + 1e-6)
        A[j, i] = A[i, j]

# self-loops
np.fill_diagonal(A, 1.0)

# row-normalize
A = A / A.sum(axis=1, keepdims=True)

A = torch.tensor(A, dtype=torch.float32)

assert torch.allclose(A.sum(dim=1), torch.ones(N)), "❌ Bad normalization"


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(6,6))
plt.scatter(coords[:,0], coords[:,1], c="red")

for i in range(N):
    for j in neighbors[i]:
        plt.plot(
            [coords[i,0], coords[j,0]],
            [coords[i,1], coords[j,1]],
            color="gray", alpha=0.5
        )

plt.title("CAF Sensor Spatial Graph (kNN + Distance)")
plt.axis("equal")
plt.grid(True)
plt.show()


# Fault Injection

In [ ]:
# ================= FAULT CONFIG =================
FAULT_SENSOR = 2
FAULT_START  = 20000
FAULT_END    = 20500
FAULT_MAG    = 5.0   # in normalized units


In [ ]:
# ================= FAULT GROUND TRUTH =================
T, N = X.shape

fault_gt = torch.zeros((T, N), dtype=torch.bool)
fault_gt[FAULT_START:FAULT_END, FAULT_SENSOR] = True

print("Total faulted samples:", fault_gt.sum().item())


In [ ]:
# ================= FAULT INJECTION =================
X_fault = X.clone()
X_fault[FAULT_START:FAULT_END, FAULT_SENSOR] += FAULT_MAG

assert torch.isfinite(X_fault).all()


In [ ]:
import matplotlib.pyplot as plt

s = FAULT_SENSOR

plt.figure(figsize=(12,4))
plt.plot(X[:, s].cpu(), label="Clean", alpha=0.6)
plt.plot(X_fault[:, s].cpu(), label="Faulty", alpha=0.8)

plt.axvspan(FAULT_START, FAULT_END, color="red", alpha=0.2)
plt.legend()
plt.title(f"Injected Fault — Sensor {s}")
plt.show()


In [ ]:
plt.figure(figsize=(12,4))
plt.imshow(
    fault_gt.T.cpu(),
    aspect="auto",
    cmap="gray_r"
)
plt.xlabel("Time")
plt.ylabel("Sensor")
plt.title("Ground Truth Fault Mask (White = Fault)")
plt.show()


In [ ]:
# ================= SPATIAL BASELINE PREDICTOR =================
A_np = A.cpu().numpy()

def spatial_predictor(X, A):
    """
    X: (T, N)
    A: (N, N) adjacency (row-normalized)
    """
    return X @ A.T


In [ ]:
# ================= PREDICTIONS & RESIDUALS =================
pred_spatial = spatial_predictor(X_fault, A)

residuals = torch.abs(X_fault - pred_spatial)
res_np = residuals.cpu().numpy()


In [ ]:
assert residuals.shape == X.shape


In [ ]:
s = FAULT_SENSOR

plt.figure(figsize=(12,4))
plt.plot(res_np[:, s], label="Residual")
plt.axvspan(FAULT_START, FAULT_END, color="red", alpha=0.2)
plt.legend()
plt.title(f"Residual Signal — Sensor {s}")
plt.show()


In [ ]:
# ================= ROBUST THRESHOLDS =================
res_np = residuals.cpu().numpy()

median = np.nanmedian(res_np, axis=0)
mad = np.nanmedian(np.abs(res_np - median), axis=0)

THRESH = median + 4.0 * mad   # conservative


In [ ]:
# ================= PROBABILISTIC FAULT DETECTION =================
eps = 1e-6
alpha = 2.0      # sharpness
beta  = 2.5      # soft threshold
lambda_mem = 0.95

res = res_np  # (T, N)

median = np.nanmedian(res, axis=0)
mad = np.nanmedian(np.abs(res - median), axis=0) + eps

# normalized residual score
z = (res - median) / mad
z = np.clip(z, 0, None)

# instantaneous fault likelihood
p_inst = 1.0 / (1.0 + np.exp(-alpha * (z - beta)))

# temporal belief propagation
p_fault = np.zeros_like(p_inst)

for t in range(T):
    if t == 0:
        p_fault[t] = p_inst[t]
    else:
        p_fault[t] = (
            lambda_mem * p_fault[t-1]
            + (1 - lambda_mem) * p_inst[t]
        )

assert np.all((p_fault >= 0) & (p_fault <= 1))


In [ ]:
plt.figure(figsize=(14,5))
plt.imshow(p_fault.T, aspect="auto", cmap="inferno")
plt.colorbar(label="P(fault)")
plt.title("Probabilistic Fault Belief")
plt.xlabel("Time")
plt.ylabel("Sensor")
plt.show()


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(14,5))
plt.imshow(
    res_np.T,
    aspect="auto",
    cmap="hot"
)
plt.colorbar(label="Absolute Residual")
plt.xlabel("Time")
plt.ylabel("Sensor")
plt.title("Residual Heatmap (Time × Sensors)")
plt.show()


In [ ]:
y_true = fault_gt.cpu().numpy().reshape(-1).astype(int)
y_score = p_fault.reshape(-1)


In [ ]:
from sklearn.metrics import roc_auc_score

auc = roc_auc_score(y_true, y_score)
print(f"ROC-AUC: {auc:.3f}")


In [ ]:
from sklearn.metrics import average_precision_score

ap = average_precision_score(y_true, y_score)
print(f"PR-AUC: {ap:.3f}")


In [ ]:
def neighbor_fault_probability(
    t,
    target,
    X_obs,
    X_temp_hat,
    p_fault,
    nbr_idx,
    health,
    alpha=2.0,
    beta=1.5,
    eps=1e-6
):
    nbrs = nbr_idx[target]
    x_nbr = X_obs[t, nbrs]

    # disagreement with temporal belief
    d_temp = torch.abs(x_nbr - X_temp_hat[t, target])

    # disagreement with neighbor consensus
    consensus = torch.median(X_obs[t, nbrs], dim=0).values
    d_cons = torch.abs(x_nbr - consensus)

    # contextual gating
    w_temp = 1.0 - p_fault[t, target]
    w_cons = 1.0

    # robust scaling
    scale = torch.median(d_cons) + eps
    z = (w_temp * d_temp + w_cons * d_cons) / scale

    # likelihood
    p = torch.sigmoid(alpha * (z - beta))

    # long-term health prior
    p = p * (1.0 - health[nbrs])

    return torch.clamp(p, 0.0, 1.0)


In [ ]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader


In [ ]:
# ========= GLOBAL CONFIG =========
sensor_idx = 2
W = 24
t_start = 20000
t_end = 20500
# ================================



# Temporal Only

In [ ]:
class TemporalDataset(Dataset):
    def __init__(self, series, window):
        self.series = series
        self.window = window

    def __len__(self):
        return len(self.series) - self.window

    def __getitem__(self, i):
        x = self.series[i:i+self.window]
        y = self.series[i+self.window]
        return x.unsqueeze(-1), y


In [ ]:
# ================= CLEAN & FAULT DATA =================

# Clean (normalized, no fault)
X_clean = X.clone()          # X was your normalized df_filled tensor

# Faulty (normalized, injected fault)
X_fault = X_fault.clone()    # from inject_bias(...)


In [ ]:
print(X_clean.shape, X_fault.shape)
assert X_clean.shape == X_fault.shape


In [ ]:
train_ds = TemporalDataset(X_clean[:, sensor_idx], W)
test_ds  = TemporalDataset(X_fault[:, sensor_idx], W)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
test_loader  = DataLoader(test_ds, batch_size=64, shuffle=False)


In [ ]:
class TemporalGRU(nn.Module):
    def __init__(self, hidden=32):
        super().__init__()
        self.gru = nn.GRU(1, hidden, batch_first=True)
        self.fc = nn.Linear(hidden, 1)

    def forward(self, x):
        _, h = self.gru(x)
        return self.fc(h.squeeze(0)).squeeze(-1)


In [ ]:
model = TemporalGRU()
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()


In [ ]:
for epoch in range(20):
    total = 0
    for x, y in train_loader:
        pred = model(x)
        loss = loss_fn(pred, y)

        opt.zero_grad()
        loss.backward()
        opt.step()

        total += loss.item()

    print(f"Epoch {epoch+1}: {total/len(train_loader):.6f}")


In [ ]:
model.eval()
preds = []

with torch.no_grad():
    for x, _ in test_loader:
        preds.append(model(x))

preds = torch.cat(preds)


In [ ]:
# ================= TEMPORAL RECONSTRUCTION (CORRECT) =================

T, N = X_fault.shape
X_temp_hat = X_fault.clone()   # fallback is faulty signal

# preds shape: (T-W,) for single sensor
X_temp_hat[W:, sensor_idx] = preds.squeeze()

# sanity checks
assert X_temp_hat.shape == X_fault.shape
assert not torch.isnan(X_temp_hat).any()


In [ ]:
true_clean = X_clean[W:, sensor_idx]
faulty_obs = X_fault[W:, sensor_idx]

rmse_faulty = torch.sqrt(((faulty_obs - true_clean)**2).mean())
rmse_temp   = torch.sqrt(((preds - true_clean)**2).mean())

print("RMSE faulty:", rmse_faulty.item())
print("RMSE temporal:", rmse_temp.item())


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(14,4))
plt.plot(true_clean, label="True clean", alpha=0.6)
plt.plot(faulty_obs, label="Faulty", alpha=0.4)
plt.plot(preds, label="Temporal reconstruction", linewidth=2)

plt.axvspan(t_start-W, t_end-W, color="red", alpha=0.2)
plt.legend()
plt.title("Temporal Reconstruction (Single Sensor)")
plt.show()


In [ ]:
# ================= FAULT WINDOW MASK =================
mask_fault_window = torch.zeros(T, dtype=torch.bool)
mask_fault_window[FAULT_START:FAULT_END] = True

assert mask_fault_window.shape[0] == T


In [ ]:
# ================= GROUND TRUTH =================
Y_true = X_clean

assert Y_true.shape == X_fault.shape


In [ ]:
rmse_temporal_fault_window = torch.sqrt(
    torch.mean(
        (X_temp_hat[mask_fault_window, sensor_idx]
         - Y_true[mask_fault_window, sensor_idx]) ** 2
    )
)


# Spatial Only

In [ ]:
import numpy as np

N = coords.shape[0]

# Pairwise distances
dist = np.linalg.norm(coords[:, None, :] - coords[None, :, :], axis=2)

# Avoid self-edges
np.fill_diagonal(dist, np.inf)

# Hyperparameters
K = 4            # number of neighbors
eps = 1e-6       # numerical stability

# Nearest neighbors per sensor
nbr_idx = np.argsort(dist, axis=1)[:, :K]   # (N, K)
nbr_dist = np.take_along_axis(dist, nbr_idx, axis=1)

# Inverse-distance weights
W_spatial = 1.0 / (nbr_dist + eps)
W_spatial = W_spatial / W_spatial.sum(axis=1, keepdims=True)


In [ ]:
X_spatial_hat = spatial_reconstruct(
    X_fault.numpy(),
    fault_flags,
    nbr_idx,
    W_spatial
)

X_spatial_hat = torch.tensor(X_spatial_hat, dtype=torch.float32)


In [ ]:
def spatial_reconstruct(
    X_obs,
    X_temp_hat,
    p_fault,
    nbr_idx,
    nbr_dist,
    health,
    eps=1e-6
):
    """
    Belief-based spatial reconstruction (Step 2 compliant)
    """
    T, N = X_obs.shape
    X_hat = X_obs.clone()

    inv_dist = 1.0 / (nbr_dist + eps)  # (N, K)

    for t in range(T):
        for s in range(N):

            nbrs = nbr_idx[s]
            geo_w = inv_dist[s]

            # ---- neighbor fault belief ----
            p_nbr = neighbor_fault_probability(
                t,
                s,
                X_obs,
                X_temp_hat,
                p_fault,
                nbr_idx,
                health
            )

            trust_nbr = 1.0 - p_nbr

            # ---- spatial centroid ----
            w = geo_w * trust_nbr
            w = w / (w.sum() + eps)

            spatial_est = torch.sum(w * X_obs[t, nbrs])

            # ---- belief-aware application ----
            pf = p_fault[t, s]

            X_hat[t, s] = (
                (1.0 - pf) * X_obs[t, s]
                + pf * spatial_est
            )

    return X_hat


In [ ]:
# True clean reference
Y_true = X_clean

# RMSE over full signal
rmse_spatial = torch.sqrt(
    torch.mean((X_spatial_hat - Y_true) ** 2)
)

print("RMSE spatial-only:", rmse_spatial.item())


In [ ]:
mask_fault_window = torch.zeros(Y_true.shape[0], dtype=torch.bool)
mask_fault_window[t_start:t_end] = True

rmse_spatial_fault_window = torch.sqrt(
    torch.mean(
        (X_spatial_hat[mask_fault_window, sensor_idx]
         - Y_true[mask_fault_window, sensor_idx]) ** 2
    )
)

rmse_faulty_fault_window = torch.sqrt(
    torch.mean(
        (X_fault[mask_fault_window, sensor_idx]
         - Y_true[mask_fault_window, sensor_idx]) ** 2
    )
)

print("RMSE faulty (fault window):", rmse_faulty_fault_window.item())
print("RMSE spatial (fault window):", rmse_spatial_fault_window.item())


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(14,4))
plt.plot(Y_true[:, sensor_idx], label="True clean", alpha=0.6)
plt.plot(X_fault[:, sensor_idx], label="Faulty", alpha=0.4)
plt.plot(X_spatial_hat[:, sensor_idx], label="Spatial reconstruction", linewidth=2)

plt.axvspan(t_start, t_end, color="red", alpha=0.2)
plt.legend()
plt.title("Spatial-only Reconstruction (Single Sensor)")
plt.show()


In [ ]:
rmse_spatial_fault_window = torch.sqrt(
    torch.mean(
        (X_spatial_hat[mask_fault_window, sensor_idx]
         - Y_true[mask_fault_window, sensor_idx]) ** 2
    )
)


# Rule-based Temporal + Spatial

In [ ]:
def compute_trust_weights(fault_len, max_len=200):
    """
    fault_len: how long the current fault has lasted
    """
    # Temporal trust decays with fault duration
    alpha_temp = np.exp(-fault_len / max_len)
    alpha_spatial = 1.0 - alpha_temp
    return alpha_temp, alpha_spatial


In [ ]:
def fuse_reconstruction(
    X_fault,
    X_temp_hat,
    X_spatial_hat,
    p_fault,
    p_fault_threshold=0.6,
    eps=1e-6
):
    T, N = X_fault.shape
    X_fused = X_fault.clone()

    fault_duration = np.zeros(N, dtype=int)

    for t in range(T):
        for s in range(N):

            pf = p_fault[t, s]

            # update duration softly
            if pf > p_fault_threshold:
                fault_duration[s] += 1
            else:
                fault_duration[s] = 0

            # temporal vs spatial trust (slow dynamics)
            a_t, a_s = compute_trust_weights(fault_duration[s])

            # combined reconstructed belief
            recon = (
                a_t * X_temp_hat[t, s]
                + a_s * X_spatial_hat[t, s]
            )

            # 🔑 belief-aware fusion
            X_fused[t, s] = (
                (1.0 - pf) * X_fault[t, s]
                + pf * recon
            )

    return X_fused


In [ ]:
X_fused = fuse_reconstruction(
    X_fault,
    X_temp_hat,
    X_spatial_hat,
    p_fault          # ✅ CORRECT
)


In [ ]:
rmse_fused_fault_window = torch.sqrt(
    torch.mean(
        (X_fused[mask_fault_window, sensor_idx]
         - Y_true[mask_fault_window, sensor_idx]) ** 2
    )
)


In [ ]:
print("RMSE temporal (fault window):", rmse_temporal_fault_window.item())
print("RMSE spatial  (fault window):", rmse_spatial_fault_window.item())
print("RMSE fused    (fault window):", rmse_fused_fault_window.item())


In [ ]:
plt.figure(figsize=(14,4))
plt.plot(Y_true[:, sensor_idx], label="True clean", alpha=0.6)
plt.plot(X_fault[:, sensor_idx], label="Faulty", alpha=0.3)
plt.plot(X_temp_hat[:, sensor_idx], label="Temporal", linewidth=2)
plt.plot(X_spatial_hat[:, sensor_idx], label="Spatial", linewidth=2)
plt.plot(X_fused[:, sensor_idx], label="Fused", linewidth=2)

plt.axvspan(t_start, t_end, color="red", alpha=0.2)
plt.legend()
plt.title("Temporal vs Spatial vs Fused Reconstruction")
plt.show()


# Trust-Weighted Fusion

In [ ]:
A_bin = (A > 0).float()
neighbors = {
    i: torch.where(A_bin[i])[0]
    for i in range(A.shape[0])
}

In [ ]:
assert X_temp_hat.shape == X_fault.shape
assert X_spatial_hat.shape == X_fault.shape
assert X_fused.shape == X_fault.shape


In [ ]:
# ================= TRUST SCORES (SAFE, SELF-CONTAINED) =================
eps = 1e-6

diff = torch.abs(X_fault - X_temp_hat)
scale = torch.median(diff, dim=0).values + eps

trust = torch.exp(-diff / scale)
trust = torch.clamp(trust, min=1e-6, max=1.0)

assert trust.shape == X_fault.shape
assert torch.isfinite(trust).all()

print(
    "trust stats →",
    "min:", trust.min().item(),
    "max:", trust.max().item(),
    "mean:", trust.mean().item()
)


In [ ]:
rmse_temporal_fault_window = torch.sqrt(
    torch.mean(
        (X_temp_hat[mask_fault_window, sensor_idx]
         - Y_true[mask_fault_window, sensor_idx]) ** 2
    )
)

rmse_spatial_fault_window = torch.sqrt(
    torch.mean(
        (X_spatial_hat[mask_fault_window, sensor_idx]
         - Y_true[mask_fault_window, sensor_idx]) ** 2
    )
)

rmse_fused_fault_window = torch.sqrt(
    torch.mean(
        (X_fused[mask_fault_window, sensor_idx]
         - Y_true[mask_fault_window, sensor_idx]) ** 2
    )
)

print("RMSE temporal (fault window):", rmse_temporal_fault_window.item())
print("RMSE spatial  (fault window):", rmse_spatial_fault_window.item())
print("RMSE fused    (fault window):", rmse_fused_fault_window.item())


# Fault Recovery Dynamics

In [ ]:
fault_flags_vis = p_fault > p_fault_threshold


In [ ]:
recovery_start = torch.zeros((T, N), dtype=torch.bool)

for i in range(N):
    for t in range(1, T):

        pf_prev = p_fault[t-1, i]
        pf_now  = p_fault[t, i]

        recovery_start[t, i] = (
            (pf_prev > p_fault_threshold) &
            (pf_now  <= p_fault_threshold)
        )


In [ ]:
recovery_age = torch.zeros((T, N), dtype=torch.long)

for i in range(N):
    for t in range(1, T):

        pf_now  = p_fault[t, i]
        pf_prev = p_fault[t-1, i]

        # recovery starts when belief crosses downward
        if pf_prev > p_fault_threshold and pf_now <= p_fault_threshold:
            recovery_age[t, i] = 1

        # recovery continues while belief stays low
        elif recovery_age[t-1, i] > 0 and pf_now <= p_fault_threshold:
            recovery_age[t, i] = recovery_age[t-1, i] + 1

        else:
            recovery_age[t, i] = 0


In [ ]:
RECOVERY_HORIZON = 24   # e.g. 1 day if hourly


In [ ]:
def recovery_aware_fusion(
    X_fault,
    X_temp_hat,
    X_spatial_hat,
    p_fault,
    recovery_age,
    recovery_horizon=24,
    eps=1e-6
):
    T, N = X_fault.shape
    X_fused = X_fault.clone()

    for t in range(T):
        for i in range(N):

            pf = p_fault[t, i]

            # ---------- FAULT (HIGH CONFIDENCE) ----------
            if pf > 0.7:
                X_fused[t, i] = X_spatial_hat[t, i]
                continue

            # ---------- RECOVERY ----------
            if recovery_age[t, i] > 0:

                p_rec = min(recovery_age[t, i] / recovery_horizon, 1.0)

                # disagreements
                e_temp = torch.abs(X_temp_hat[t, i] - X_spatial_hat[t, i])
                e_sensor = torch.abs(X_fault[t, i] - X_spatial_hat[t, i])

                trust_temp = torch.exp(-e_temp)
                trust_sensor = torch.exp(-e_sensor)

                # safety gate
                if e_sensor > 2.0 * e_temp:
                    trust_sensor = torch.tensor(
                        0.0, device=X_fault.device
                    )

                # belief-aware weights
                w_spatial = 0.5 * pf
                w_temp    = 0.3 * p_rec * trust_temp * (1.0 - pf)
                w_sensor  = 0.2 * p_rec * trust_sensor * (1.0 - pf)

                Z = w_spatial + w_temp + w_sensor + eps
                w_spatial /= Z
                w_temp    /= Z
                w_sensor  /= Z

                X_fused[t, i] = (
                    w_spatial * X_spatial_hat[t, i]
                    + w_temp    * X_temp_hat[t, i]
                    + w_sensor  * X_fault[t, i]
                )

            # ---------- HEALTHY ----------
            else:
                X_fused[t, i] = (
                    (1.0 - pf) * X_fault[t, i]
                    + pf * X_spatial_hat[t, i]
                )

    return X_fused


In [ ]:
X_fused_recovery = recovery_aware_fusion(
    X_fault,
    X_temp_hat,
    X_spatial_hat,
    p_fault,          # ✅ CORRECT
    recovery_age,
    RECOVERY_HORIZON
)


In [ ]:
recovery_mask = (
    (recovery_age[:, sensor_idx] > 0) &
    (recovery_age[:, sensor_idx] <= RECOVERY_HORIZON)
)

rmse_recovery = torch.sqrt(
    torch.mean(
        (X_fused_recovery[recovery_mask, sensor_idx]
         - Y_true[recovery_mask, sensor_idx]) ** 2
    )
)

rmse_spatial_recovery = torch.sqrt(
    torch.mean(
        (X_spatial_hat[recovery_mask, sensor_idx]
         - Y_true[recovery_mask, sensor_idx]) ** 2
    )
)

print("RMSE recovery (fused):", rmse_recovery.item())
print("RMSE recovery (spatial):", rmse_spatial_recovery.item())


In [ ]:
plt.figure(figsize=(14,4))
plt.plot(Y_true[:, sensor_idx], label="True clean", alpha=0.6)
plt.plot(X_spatial_hat[:, sensor_idx], label="Spatial", alpha=0.6)
plt.plot(X_temp_hat[:, sensor_idx], label="Temporal", alpha=0.6)
plt.plot(X_fused_recovery[:, sensor_idx], label="Recovery-aware fused", linewidth=2)

plt.axvspan(FAULT_START, FAULT_END, color="red", alpha=0.2)
plt.legend()
plt.title("Fault → Recovery Dynamics")
plt.show()


# Trust-Weighted Spatial Centroid

In [ ]:
def compute_trust_scores(X_obs, X_temp_hat, health, eps=1e-6):
    diff = torch.abs(X_obs - X_temp_hat)

    scale = torch.median(diff, dim=0).values + eps
    agreement = torch.exp(-diff / scale)

    # 🔑 NEW: factor in health
    trust = agreement * health.unsqueeze(0)

    return torch.clamp(trust, min=1e-3, max=1.0)


In [ ]:
# ================= TRUST SCORES =================
trust = compute_trust_scores(
    X_fault,
    X_temp_hat,
    health        # ✅ NEW ARGUMENT
)

assert trust.shape == X_fault.shape
assert torch.isfinite(trust).all()


In [ ]:
# ================= CONVERT NEIGHBORS TO TORCH =================
nbr_idx_torch = torch.tensor(nbr_idx, dtype=torch.long)
nbr_dist_torch = torch.tensor(nbr_dist, dtype=torch.float32)

assert nbr_idx_torch.shape == nbr_dist_torch.shape


In [ ]:
X_spatial_hat = spatial_reconstruct(
    X_fault,
    X_temp_hat,
    p_fault,
    nbr_idx_torch,
    nbr_dist_torch,
    health
)


In [ ]:
plt.figure(figsize=(12,4))
for n in nbr_idx[sensor_idx]:
    plt.plot(trust[:, n].cpu(), alpha=0.6, label=f"nbr {n}")

plt.axvspan(FAULT_START, FAULT_END, color="red", alpha=0.2)
plt.title("Neighbor Trust Scores Over Time")
plt.legend()
plt.show()


In [ ]:
rmse_spatial_plain = torch.sqrt(
    torch.mean(
        (X_spatial_hat[mask_fault_window, sensor_idx]
         - Y_true[mask_fault_window, sensor_idx]) ** 2
    )
)

rmse_spatial_trust = torch.sqrt(
    torch.mean(
        (X_spatial_hat_trust[mask_fault_window, sensor_idx]
         - Y_true[mask_fault_window, sensor_idx]) ** 2
    )
)

print("RMSE spatial (distance only):", rmse_spatial_plain.item())
print("RMSE spatial (trust-weighted):", rmse_spatial_trust.item())


In [ ]:
plt.figure(figsize=(14,4))
plt.plot(Y_true[:, sensor_idx], label="True clean", alpha=0.6)
plt.plot(X_spatial_hat[:, sensor_idx], label="Spatial plain", alpha=0.6)
plt.plot(X_spatial_hat_trust[:, sensor_idx], label="Spatial trust", linewidth=2)

plt.axvspan(FAULT_START, FAULT_END, color="red", alpha=0.2)
plt.legend()
plt.title("Spatial Reconstruction: Distance vs Trust-Weighted")
plt.show()


# Inject a Faulty Neighbor (Controlled Stress Test)

In [ ]:
# =========================================================
# MULTI-NEIGHBOR SIMULTANEOUS FAULT INJECTION
# =========================================================

target = sensor_idx
neighbors_target = nbr_idx[target]   # (K,)

# ---- CONFIG ----
NUM_FAULTY_NEIGHBORS = 2   # try 2, then 3
NEIGHBOR_FAULT_MAG = 5.0

# select which neighbors fail (closest ones, worst case)
faulty_neighbors = neighbors_target[:NUM_FAULTY_NEIGHBORS]

print("Target sensor:", target)
print("Faulty neighbors:", faulty_neighbors.tolist())

# ---- inject faults ----
X_fault_nb = X_fault.clone()

for n in faulty_neighbors:
    X_fault_nb[FAULT_START:FAULT_END, n] += NEIGHBOR_FAULT_MAG

assert torch.isfinite(X_fault_nb).all()


In [ ]:
# ================= TRUST (WITH FAULTY NEIGHBOR) =================
diff = torch.abs(X_fault_nb - X_temp_hat)
scale = torch.median(diff, dim=0).values + 1e-6

trust_nb = torch.exp(-diff / scale)
trust_nb = torch.clamp(trust_nb, min=1e-6, max=1.0)

print(
    "trust stats (neighbor-fault case) →",
    trust_nb.min().item(),
    trust_nb.mean().item()
)


In [ ]:
X_spatial_nb = spatial_reconstruct(
    X_fault_nb,
    X_temp_hat,
    p_fault,
    nbr_idx_torch,
    nbr_dist_torch,
    health
)


In [ ]:
rmse_plain_nb = torch.sqrt(
    torch.mean(
        (X_spatial_plain_nb[mask_fault_window, sensor_idx]
         - Y_true[mask_fault_window, sensor_idx]) ** 2
    )
)

rmse_trust_nb = torch.sqrt(
    torch.mean(
        (X_spatial_trust_nb[mask_fault_window, sensor_idx]
         - Y_true[mask_fault_window, sensor_idx]) ** 2
    )
)

print("RMSE spatial (distance-only, neighbor faulty):", rmse_plain_nb.item())
print("RMSE spatial (trust-weighted, neighbor faulty):", rmse_trust_nb.item())


In [ ]:
plt.figure(figsize=(14,4))
plt.plot(Y_true[:, sensor_idx], label="True clean", alpha=0.6)
plt.plot(X_spatial_plain_nb[:, sensor_idx], label="Spatial plain (polluted)", alpha=0.6)
plt.plot(X_spatial_trust_nb[:, sensor_idx], label="Spatial trust (robust)", linewidth=2)

plt.axvspan(FAULT_START, FAULT_END, color="red", alpha=0.2)
plt.legend()
plt.title("Neighbor Fault Stress Test: Plain vs Trust-Weighted")
plt.show()


In [ ]:
# average trust during fault window
avg_trust_fault = trust_nb[mask_fault_window].mean(dim=0)

for n in nbr_idx[sensor_idx]:
    print(f"Neighbor {n}: avg trust = {avg_trust_fault[n].item():.3f}")


In [ ]:
for n in faulty_neighbors:
    delta = (
        X_fault_nb[FAULT_START:FAULT_END, n]
        - X_fault[FAULT_START:FAULT_END, n]
    ).mean()
    print(f"Injected delta for neighbor {n}: {delta.item():.2f}")


In [ ]:
plt.figure(figsize=(12,4))
for j in nbr_idx[sensor_idx]:
    probs = []
    for t in range(T):
        p = neighbor_fault_probability(
            t,
            sensor_idx,
            X_fault_nb,
            X_temp_hat,
            p_fault,
            nbr_idx_torch,
            health
        )
        probs.append(p[nbr_idx[sensor_idx] == j].item())

    plt.plot(probs, label=f"nbr {j}")

plt.axvspan(FAULT_START, FAULT_END, color="red", alpha=0.2)
plt.legend()
plt.title("Neighbor Fault Probability (Step 2)")
plt.show()


# Trust via RL

In [ ]:
# class NeighborFaultClassifier(nn.Module):
#     def __init__(self, state_dim=5):
#         super().__init__()
#         self.net = nn.Sequential(
#             nn.Linear(state_dim, 32),
#             nn.ReLU(),
#             nn.Linear(32, 1)  # logits
#         )

#     def forward(self, state):
#         return torch.sigmoid(self.net(state))  # P(neighbor faulty)


In [ ]:
# class TrustPolicy(nn.Module):
#     def __init__(self, state_dim=4):   # ✅ was 5
#         super().__init__()
#         self.net = nn.Sequential(
#             nn.Linear(state_dim, 32),
#             nn.ReLU(),
#             nn.Linear(32, 1),
#             nn.Sigmoid()
#         )

#     def forward(self, state):
#         return self.net(state)


In [ ]:
# def build_trust_state(
#     X_obs,
#     X_temp_hat,
#     X_spatial_hat,
#     fault_flags,
#     nbr_idx_torch
# ):
#     T, N = X_obs.shape
#     state = torch.zeros((T, N, 4), device=X_obs.device)

#     # 1. temporal disagreement
#     state[..., 0] = torch.abs(X_obs - X_temp_hat)

#     # 2. spatial disagreement
#     for i in range(N):
#         nbrs = nbr_idx_torch[i]
#         state[:, i, 1] = torch.mean(
#             torch.abs(X_obs[:, i].unsqueeze(1) - X_obs[:, nbrs]),
#             dim=1
#         )

#     # 3. neighbor dispersion
#     for i in range(N):
#         nbrs = nbr_idx_torch[i]
#         state[:, i, 2] = torch.std(X_obs[:, nbrs], dim=1)

#     # 4. fault flag
#     state[..., 3] = fault_flags.float()

#     return state


In [ ]:
# def build_neighbor_state(
#     t,
#     target,
#     X_obs,
#     X_temp_hat,
#     fault_flags,
#     nbr_idx_torch
# ):
#     """
#     Returns:
#         state: (K, 5)
#     """
#     nbrs = nbr_idx_torch[target]

#     K = len(nbrs)
#     state = torch.zeros((K, 5), device=X_obs.device)

#     # 1. temporal inconsistency (target)
#     temp_err = torch.abs(X_obs[t, target] - X_temp_hat[t, target])
#     state[:, 0] = temp_err

#     # 2. neighbor vs temporal
#     state[:, 1] = torch.abs(X_obs[t, nbrs] - X_temp_hat[t, target])

#     # 3. neighbor deviation from neighbor mean
#     nbr_vals = X_obs[t, nbrs]
#     state[:, 2] = torch.abs(nbr_vals - nbr_vals.mean())

#     # 4. neighbor dispersion
#     state[:, 3] = torch.std(nbr_vals)

#     # 5. fault flag of target
#     # state[:, 4] = fault_flags[t, target].float()

#     return state


In [ ]:
# # ================= SENSOR-LEVEL HEURISTIC TRUST (TEACHER) =================
# diff = torch.abs(X_fault - X_temp_hat)     # (T, N)
# scale = torch.median(diff, dim=0).values + 1e-6

# trust_teacher = torch.exp(-diff / scale)
# trust_teacher = torch.clamp(trust_teacher, min=1e-6, max=1.0)

# print("trust_teacher shape:", trust_teacher.shape)


In [ ]:
# state = build_trust_state(
#     X_fault,          # single-neighbor training data
#     X_temp_hat,
#     X_spatial_hat,
#     fault_flags_t,
#     nbr_idx_torch
# )


In [ ]:
# X_train = state.reshape(-1, state.shape[-1])   # (T·N, 4)
# y_train = trust_teacher.reshape(-1, 1)         # (T·N, 1)

# assert X_train.shape[0] == y_train.shape[0]


In [ ]:
# policy = TrustPolicy(state_dim=4)
# optimizer = torch.optim.Adam(policy.parameters(), lr=1e-3)

# def weighted_mse(pred, target):
#     # emphasize low-trust samples
#     weights = 1.0 + 4.0 * (target < 0.5).float()
#     return torch.mean(weights * (pred - target) ** 2)


# for epoch in range(10):
#     optimizer.zero_grad()

#     pred = policy(X_train)
#     loss = weighted_mse(pred, y_train)


#     loss.backward()
#     optimizer.step()

#     print(f"Epoch {epoch+1}: loss = {loss.item():.6f}")


In [ ]:
# with torch.no_grad():
#     trust_policy = policy(X_train).reshape(T, N)

# trust_policy = torch.clamp(trust_policy, 1e-6, 1.0)

# print("Heuristic trust mean:", trust.mean().item())
# print("Policy trust mean:", trust_policy.mean().item())
# print("Policy trust min:", trust_policy.min().item())


In [ ]:
# print("trust_teacher:", trust_teacher.shape)
# print("trust_policy:", trust_policy.shape)


In [ ]:
# plt.figure(figsize=(12,4))

# for n in faulty_neighbors:
#     plt.plot(
#         trust_teacher[:, n].cpu(),
#         label=f"Heuristic nbr {n}",
#         alpha=0.7
#     )
#     plt.plot(
#         trust_policy[:, n].detach().cpu(),
#         linestyle="--",
#         label=f"Policy nbr {n}",
#         alpha=0.7
#     )

# plt.axvspan(FAULT_START, FAULT_END, color="red", alpha=0.2)
# plt.legend()
# plt.title("Trust: Heuristic vs Learned Policy (Faulty Neighbors)")
# plt.show()


In [ ]:
# X_spatial_hat_policy = spatial_reconstruct_trust_weighted(
#     X_fault_nb,
#     fault_flags_t,
#     nbr_idx_torch,
#     nbr_dist_torch,
#     trust_policy
# )


In [ ]:
# print("Heuristic trust mean:", trust.mean().item())
# print("Policy trust mean:", trust_policy.mean().item())


# Reward based RL

In [ ]:
# policy = NeighborFaultClassifier()
# optimizer = torch.optim.Adam(policy.parameters(), lr=1e-3)


In [ ]:
# EPOCHS = 30
# eps = 1e-6

# for epoch in range(EPOCHS):

#     total_loss = 0.0
#     steps = 0
#     trust_prev = None

#     for t in range(FAULT_START, FAULT_END):

#         if not fault_flags_t[t, sensor_idx]:
#             continue

#         state = build_neighbor_state(
#             t,
#             sensor_idx,
#             X_fault,            # single-fault training
#             X_temp_hat,
#             fault_flags_t,
#             nbr_idx_torch
#         )

#         p_fault = policy(state).squeeze()
#         trust = torch.clamp(1.0 - p_fault, 0.0, 1.0)

#         nbrs = nbr_idx_torch[sensor_idx]
#         geo_w = 1.0 / (nbr_dist_torch[sensor_idx] + eps)

#         # ---------- reconstruction ----------
#         if trust.sum() < 1e-3:
#             X_hat = X_temp_hat[t, sensor_idx]
#         else:
#             w = geo_w * trust
#             w = w / (w.sum() + eps)
#             X_hat = torch.sum(w * X_fault[t, nbrs])

#         # ---------- losses ----------
#         L_rec = (X_hat - Y_true[t, sensor_idx]) ** 2

#         temp_conf = torch.exp(
#             -torch.abs(X_fault[t, sensor_idx] - X_temp_hat[t, sensor_idx])
#         )

#         L_dis = temp_conf * torch.mean(
#             trust * torch.abs(X_fault[t, nbrs] - X_temp_hat[t, sensor_idx])
#         )

#         if trust.max() > 0.5:
#             closest = torch.argmin(nbr_dist_torch[sensor_idx])
#             L_anchor = (1.0 - trust[closest])
#         else:
#             L_anchor = 0.0

#         if trust_prev is not None:
#             L_smooth = torch.mean((trust - trust_prev) ** 2)
#         else:
#             L_smooth = 0.0

#         trust_prev = trust.detach()

#         loss = (
#             L_rec
#             + 0.1 * L_dis
#             + 0.1 * L_anchor
#             + 0.05 * L_smooth
#         )

#         optimizer.zero_grad()
#         loss.backward()
#         optimizer.step()

#         total_loss += loss.item()
#         steps += 1

#     print(f"Epoch {epoch+1:02d} | loss = {total_loss / (steps + eps):.6f}")


In [ ]:
# trust_cls = torch.ones_like(X_fault_nb)

# with torch.no_grad():
#     for t in range(T):

#         if not fault_flags_t[t, sensor_idx]:
#             continue

#         state = build_neighbor_state(
#             t,
#             sensor_idx,
#             X_fault_nb,
#             X_temp_hat,
#             fault_flags_t,
#             nbr_idx_torch
#         )

#         p_fault = policy(state).squeeze()
#         trust = 1.0 - p_fault
#         trust = torch.clamp(trust, 0.0, 1.0)

#         # anchor only if at least one neighbor seems healthy
#         if trust.max() > 0.5:
#             score = trust / (nbr_dist_torch[sensor_idx] + 1e-6)
#             anchor = torch.argmax(score)
#             trust[anchor] = 1.0

#         # allow rejecting all neighbors
#         trust = torch.clamp(trust, min=0.05)
#         trust = trust / (trust.sum() + 1e-6)


#         trust_cls[t, nbr_idx_torch[sensor_idx]] = trust


In [ ]:
# X_spatial_hat_cls = spatial_reconstruct_trust_weighted(
#     X_fault_nb,
#     fault_flags_t,
#     nbr_idx_torch,
#     nbr_dist_torch,
#     trust_cls
# )


In [ ]:
# rmse_cls = torch.sqrt(
#     torch.mean(
#         (X_spatial_hat_cls[mask_fault_window, sensor_idx]
#          - Y_true[mask_fault_window, sensor_idx]) ** 2
#     )
# )

# print("RMSE (neighbor fault classifier):", rmse_cls.item())


In [ ]:
# for n in nbr_idx[sensor_idx]:
#     plt.plot(
#         trust_cls[:, n].cpu(),
#         label=f"nbr {n}",
#         alpha=0.7
#     )

# plt.axvspan(FAULT_START, FAULT_END, color="red", alpha=0.2)
# plt.legend()
# plt.title("Learned Neighbor Trust (Fault Classification)")
# plt.show()


In [ ]:
# class NeighborTrustPolicy(nn.Module):
#     def __init__(self, K):
#         super().__init__()
#         self.net = nn.Sequential(
#             nn.Linear(5, 32),
#             nn.ReLU(),
#             nn.Linear(32, K)
#         )

#     def forward(self, state):
#         return torch.softmax(self.net(state), dim=-1)


In [ ]:
# def reconstruction_loss(
#     X_hat,
#     Y_true,
#     mask_fault_window,
#     sensor_idx
# ):
#     """
#     Negative reward = MSE over fault window
#     """
#     return torch.mean(
#         (X_hat[mask_fault_window, sensor_idx]
#          - Y_true[mask_fault_window, sensor_idx]) ** 2
#     )


In [ ]:
# def neighbor_disagreement_loss(
#     X_temp_hat,
#     X_spatial_hat,
#     trust,
#     fault_flags,
#     nbr_idx,
#     sensor_idx,
#     eps=1e-6
# ):
#     """
#     Penalize trusting neighbors that disagree with temporal model
#     Only active during fault window of target sensor
#     """
#     T = X_temp_hat.shape[0]
#     loss_terms = []

#     for t in range(T):
#         if not fault_flags[t, sensor_idx]:
#             continue

#         nbrs = nbr_idx[sensor_idx]  # (K,)

#         # disagreement of each neighbor with temporal belief
#         err = torch.abs(
#             X_spatial_hat[t, nbrs] - X_temp_hat[t, sensor_idx]
#         )  # (K,)

#         trust_w = trust[t, nbrs]    # (K,)

#         # penalize trusting high-error neighbors
#         loss_terms.append(
#             torch.mean(trust_w * err)
#         )

#     if len(loss_terms) == 0:
#         return torch.tensor(0.0, device=X_temp_hat.device)

#     return torch.mean(torch.stack(loss_terms))


In [ ]:
# policy = TrustPolicy()
# optimizer = torch.optim.Adam(policy.parameters(), lr=1e-4)

# for epoch in range(30):

#     # ---- build state ----
#     state = build_trust_state(
#         X_fault,
#         X_temp_hat,
#         X_spatial_hat,
#         fault_flags_t,
#         trust_prev=torch.ones_like(X_fault),  # no teacher now
#         nbr_idx_torch=nbr_idx_torch
#     )

#     X_state = state.reshape(-1, 5)

#     # ---- policy trust ----
#     trust_rl = policy(X_state).reshape(T, N)
#     trust_rl = torch.clamp(trust_rl, 1e-6, 1.0)

#     # ---- spatial reconstruction ----
#     X_hat_rl = spatial_reconstruct_trust_weighted(
#         X_fault,
#         fault_flags_t,
#         nbr_idx_torch,
#         nbr_dist_torch,
#         trust_rl
#     )

#     penalty = neighbor_disagreement_loss(
#         X_temp_hat,
#         X_spatial_hat,
#         trust_rl,
#         fault_flags_t,
#         nbr_idx_torch,
#         sensor_idx
#     )

#     # ---- loss (negative reward) ----
#     loss = reconstruction_loss(
#         X_hat_rl,
#         Y_true,
#         mask_fault_window,
#         sensor_idx
#     ) + 0.1 * penalty

#     optimizer.zero_grad()
#     loss.backward()
#     optimizer.step()

#     print(f"Epoch {epoch+1:02d} | reconstruction MSE = {loss.item():.6f}")


In [ ]:
# plt.figure(figsize=(12,4))
# plt.plot(
#     trust_rl[:, faulty_neighbor].detach().cpu().numpy(),
#     label="RL trust"
# )
# plt.axvspan(FAULT_START, FAULT_END, color="red", alpha=0.2)
# plt.legend()
# plt.title("Learned Trust (Reward-Based)")
# plt.show()


In [ ]:
# rmse_rl = torch.sqrt(
#     reconstruction_loss(
#         X_hat_rl,
#         Y_true,
#         mask_fault_window,
#         sensor_idx
#     )
# )

# print("RMSE (RL trust):", rmse_rl.item())


In [ ]:
# for n in nbr_idx[sensor_idx]:
#     if n == faulty_neighbor:
#         continue
#     plt.plot(
#         trust_rl[:, n].detach().cpu(),
#         alpha=0.6,
#         label=f"nbr {n}"
#     )

# plt.axvspan(FAULT_START, FAULT_END, color="red", alpha=0.2)
# plt.legend()
# plt.title("RL Trust — Healthy Neighbors")
# plt.show()


In [ ]:
# outside_fault = ~mask_fault_window

# mean_trust_outside = trust_cls[outside_fault][:, nbr_idx[sensor_idx]].mean()

# print("Mean trust outside fault:", mean_trust_outside.item())


In [ ]:
# for n in nbr_idx[sensor_idx]:
#     print(
#         f"Neighbor {n} | "
#         f"mean trust (fault window): "
#         f"{trust_cls[mask_fault_window, n].mean().item():.3f}"
#     )


In [ ]:
# plt.figure(figsize=(12,4))
# for n in neighbors_target:
#     plt.plot(
#         trust_cls[:, n].cpu(),
#         label=f"nbr {n}",
#         alpha=0.7
#     )

# plt.axvspan(FAULT_START, FAULT_END, color="red", alpha=0.2)
# plt.legend()
# plt.title("Trust under Multi-Neighbor Faults")
# plt.show()
